# Analýza zarovnania notového zápisu a zvukovej nahrávky

Tento notebook slúži ako doplnkový experimentálny materiál k bakalárskej práci **Tempomapa hudobného záznamu**. Jeho úlohou je načítať výstupy implementovanej pipeline, zobraziť vybrané lokálne úseky zarovnania a uložiť grafy použiteľné v texte práce.

Notebook nevypočítava celé zarovnanie od začiatku. Predpokladá, že hlavný program už vytvoril výstupné súbory v priečinku `output/<skladba>/analysis/`, najmä tempomapu, chroma príznaky a prípadne aj zarovnávaciu cestu DTW.

**Použitá terminológia v grafoch:** notový zápis, nahrávka, tempomapa, zarovnávacia cesta a matica nákladov.


## 1. Inicializácia prostredia

V Google Colabe sa automaticky pripojí Google Drive. Pri lokálnom spustení stačí nastaviť premennú `PROJECT_ROOT` na koreňový priečinok repozitára, v ktorom sa nachádza priečinok `output`.


In [ ]:
from pathlib import Path
import os
import json
import zipfile
import shutil
import warnings

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Audio

# Notebook funguje v Google Colabe aj v lokálnom Jupyter prostredí.
# V Colabe sa automaticky pripojí Google Drive; lokálne stačí upraviť cestu PROJECT_ROOT v ďalšej bunke.
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive
    MOUNT_POINT = "/content/gdrive"
    drive.mount(MOUNT_POINT, force_remount=True)
    DEFAULT_PROJECT_ROOT = Path(MOUNT_POINT) / "MyDrive" / "bakalarka"
else:
    DEFAULT_PROJECT_ROOT = Path.cwd()

print("Prostredie:", "Google Colab" if IN_COLAB else "lokálne Jupyter prostredie")
print("Predvolený koreň projektu:", DEFAULT_PROJECT_ROOT)


## 2. Konfigurácia notebooku

Skontroluj najmä `PROJECT_ROOT`. Ak máš repozitár alebo výstupy uložené inde, uprav túto cestu pred spustením ďalších buniek.


In [ ]:
# @title Nastavenia ciest a načítania
PROJECT_ROOT = DEFAULT_PROJECT_ROOT
OUTPUT_ROOT = PROJECT_ROOT / 'output'
LOAD_CHROMAS_AT_START = True
PREFERRED_DEFAULT_SONG = 'chopin'

TEMPOMAP_FILE_NAMES = [
    'tempomap_smooth.npy',
    'tempomap_smoothed.npy',
    'smooth_tempomap.npy',
    'tempomap.npy',
    'tempomap_raw.npy',
    'raw_tempomap.npy',
]


def has_any_file(base_dir, names):
    return any(((base_dir / name).exists() for name in names))


def find_song_dirs(output_root=OUTPUT_ROOT):
    if not output_root.exists():
        raise FileNotFoundError(
            f'Nenašiel som OUTPUT_ROOT: {output_root}\n'
            'Skontroluj, či je správne nastavený PROJECT_ROOT a či už boli vygenerované výstupy pipeline.'
        )
    song_dirs = {}
    for song_root in sorted(output_root.iterdir()):
        if not song_root.is_dir():
            continue
        candidates = [song_root / 'analysis', song_root]
        for candidate in candidates:
            if candidate.exists() and has_any_file(candidate, TEMPOMAP_FILE_NAMES):
                song_dirs[song_root.name] = candidate
                break
    if not song_dirs:
        raise FileNotFoundError(
            f'V priečinku {output_root} som nenašiel žiadnu skladbu s tempomapou.\n'
            'Očakávaný príklad: output/chopin/analysis/tempomap_smooth.npy'
        )
    return song_dirs


SONG_DIRS = find_song_dirs()
print('Nájdené skladby:')
for song, path in SONG_DIRS.items():
    print(f' - {song}: {path}')


## 3. Načítanie výstupov pipeline

Táto časť vyhľadá výstupy hlavnej implementácie pre jednotlivé skladby. Notebook očakáva, že zarovnanie už bolo vypočítané a že v priečinku `output/<skladba>/analysis/` existuje tempomapa, chroma príznaky a prípadne aj zarovnávacia cesta DTW.


In [ ]:
# @title Načítanie dát a pomocné funkcie
FILE_ALIASES = {'tempomap': TEMPOMAP_FILE_NAMES, 'audio_chroma': ['audio_chroma.npy', 'chroma_audio.npy'], 'score_chroma': ['score_chroma.npy', 'chroma_score.npy'], 'audio_times': ['audio_times.npy', 'audio_frame_times.npy', 'times_audio.npy'], 'score_times': ['score_times.npy', 'score_frame_times.npy', 'times_score.npy'], 'metadata': ['metadata.json', 'analysis_metadata.json', 'config.json'], 'audio': ['audio_full.wav', 'audio.wav', 'performance.wav'], 'path_frames': ['path_frames.npy', 'warping_path_frames.npy', 'dtw_path_frames.npy', 'path.npy', 'warping_path.npy'], 'score_chroma_on_audio': ['score_chroma_on_audio_time.npy', 'score_chroma_warped_to_audio.npy', 'score_chroma_audio_time.npy']}

def find_first_existing(base_dir, names):
    for name in names:
        path = base_dir / name
        if path.exists():
            return path
    return None

def load_npy(base_dir, key, required=False):
    path = find_first_existing(base_dir, FILE_ALIASES[key])
    if path is None:
        if required:
            raise FileNotFoundError(f"Chýba súbor pre '{key}'. Očakávané názvy: {FILE_ALIASES[key]}")
        return (None, None)
    return (np.load(path, allow_pickle=True), path)

def load_json(base_dir, key, required=False):
    path = find_first_existing(base_dir, FILE_ALIASES[key])
    if path is None:
        if required:
            raise FileNotFoundError(f"Chýba JSON/CSV súbor pre '{key}'. Očakávané názvy: {FILE_ALIASES[key]}")
        return (None, None)
    with open(path, 'r', encoding='utf-8') as f:
        return (json.load(f), path)

def orient_chroma(chroma):
    if chroma is None:
        return None
    arr = np.asarray(chroma, dtype=float)
    if arr.ndim != 2:
        raise ValueError(f'Chroma musí byť 2D matica, dostal som tvar {arr.shape}.')
    if arr.shape[0] == 12:
        return arr
    if arr.shape[1] == 12:
        return arr.T
    raise ValueError(f'Neviem určiť orientáciu chroma matice s tvarom {arr.shape}. Očakávam 12 binov.')

def orient_time_major_chroma(chroma):
    arr = orient_chroma(chroma)
    if arr is None:
        return None
    return arr.T

def infer_times(n_frames, metadata, kind):
    metadata = metadata or {}
    candidate_keys = [f'{kind}_hop_seconds', f'{kind}_frame_hop_seconds', 'hop_seconds', 'frame_hop_seconds', 'hop_time']
    for key in candidate_keys:
        if key in metadata:
            hop = float(metadata[key])
            return np.arange(n_frames, dtype=float) * hop
    sr = metadata.get('sr') or metadata.get('sample_rate')
    hop_length = metadata.get(f'{kind}_hop_length') or metadata.get('hop_length')
    if sr and hop_length:
        return np.arange(n_frames, dtype=float) * float(hop_length) / float(sr)
    warnings.warn(f'Chýba časová os pre {kind}. Používam indexy rámcov namiesto sekúnd/beatov. Odporúčané je exportovať {kind}_times.npy.')
    return np.arange(n_frames, dtype=float)

def load_song_data(song, song_dir, load_chromas=True):
    metadata, metadata_path = load_json(song_dir, 'metadata', required=False)
    tempomap, tempomap_path = load_npy(song_dir, 'tempomap', required=True)
    if load_chromas:
        audio_chroma, audio_chroma_path = load_npy(song_dir, 'audio_chroma', required=False)
        score_chroma, score_chroma_path = load_npy(song_dir, 'score_chroma', required=False)
        audio_times, audio_times_path = load_npy(song_dir, 'audio_times', required=False)
        score_times, score_times_path = load_npy(song_dir, 'score_times', required=False)
        audio_chroma = orient_chroma(audio_chroma)
        score_chroma = orient_chroma(score_chroma)
        if audio_chroma is not None and audio_times is None:
            audio_times = infer_times(audio_chroma.shape[1], metadata, 'audio')
        if score_chroma is not None and score_times is None:
            score_times = infer_times(score_chroma.shape[1], metadata, 'score')
    else:
        audio_chroma = score_chroma = audio_times = score_times = None
        audio_chroma_path = score_chroma_path = audio_times_path = score_times_path = None
    path_frames, path_frames_path = load_npy(song_dir, 'path_frames', required=False)
    score_chroma_on_audio, score_chroma_on_audio_path = load_npy(song_dir, 'score_chroma_on_audio', required=False)
    score_chroma_on_audio = orient_chroma(score_chroma_on_audio)
    audio_path = find_first_existing(song_dir, FILE_ALIASES['audio'])
    return {'song': song, 'song_dir': song_dir, 'metadata': metadata, 'metadata_path': metadata_path, 'tempomap': tempomap, 'tempomap_path': tempomap_path, 'audio_chroma': audio_chroma, 'audio_chroma_path': audio_chroma_path, 'score_chroma': score_chroma, 'score_chroma_path': score_chroma_path, 'audio_times': None if audio_times is None else np.asarray(audio_times, dtype=float).squeeze(), 'audio_times_path': audio_times_path, 'score_times': None if score_times is None else np.asarray(score_times, dtype=float).squeeze(), 'score_times_path': score_times_path, 'path_frames': None if path_frames is None else np.asarray(path_frames), 'path_frames_path': path_frames_path, 'score_chroma_on_audio': score_chroma_on_audio, 'score_chroma_on_audio_path': score_chroma_on_audio_path, 'audio_path': audio_path}

def load_all_songs(song_dirs=SONG_DIRS, load_chromas=LOAD_CHROMAS_AT_START):
    songs = {}
    for song, song_dir in song_dirs.items():
        try:
            songs[song] = load_song_data(song, song_dir, load_chromas=load_chromas)
        except Exception as exc:
            print(f"Preskakujem skladbu '{song}', lebo sa ju nepodarilo načítať: {exc}")
    if not songs:
        raise RuntimeError('Nepodarilo sa načítať žiadnu skladbu.')
    return songs

SONGS = load_all_songs()
if PREFERRED_DEFAULT_SONG in SONGS:
    aktuálna_SONG = PREFERRED_DEFAULT_SONG
else:
    aktuálna_SONG = next(iter(SONGS.keys()))

def list_songs():
    print('Načítané skladby:')
    for song, data in SONGS.items():
        tempomap_shape = np.asarray(data['tempomap']).shape
        audio_shape = None if data['audio_chroma'] is None else data['audio_chroma'].shape
        score_shape = None if data['score_chroma'] is None else data['score_chroma'].shape
        path_shape = None if data['path_frames'] is None else data['path_frames'].shape
        marker = ' <== aktuálna' if song == aktuálna_SONG else ''
        print(f' - {song}{marker}')
        print(f"   priečinok: {data['song_dir']}")
        print(f'   tempomap: {tempomap_shape}')
        print(f'   audio_chroma: {audio_shape}, score_chroma: {score_shape}')
        print(f'   zarovnávacia cesta: {path_shape}')
        if data['path_frames'] is None:
            print('   pozn.: path_frames.npy chýba, matica nákladov sa zobrazí bez warping path.')

def get_song_data(song=None):
    if song is None:
        song = aktuálna_SONG
    if song in SONGS:
        return SONGS[song]
    song_lower = str(song).lower()
    matches = [name for name in SONGS if name.lower() == song_lower]
    if len(matches) == 1:
        return SONGS[matches[0]]
    raise KeyError(f"Skladba '{song}' nie je načítaná. Dostupné skladby: {', '.join(SONGS.keys())}")

def select_song(song):
    global aktuálna_SONG
    data = get_song_data(song)
    aktuálna_SONG = data['song']
    print(f'Aktuálna skladba: {aktuálna_SONG}')
    return data

list_songs()


## 4. Práca s tempomapou

Tempomapa sa používa v oboch smeroch: z času notového zápisu do času nahrávky aj opačne. Pri duplicitných bodoch sa hodnoty spriemerujú a na mapovanie sa používa lineárna interpolácia.


In [ ]:
# @title Pomocné funkcie pre tempomapu
PITCH_LABELS = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']

def normalize_tempomap(tempomap):
    tm = np.asarray(tempomap, dtype=float)
    if tm.ndim != 2 or 2 not in tm.shape:
        raise ValueError(f'Tempomapa musí mať tvar (N, 2) alebo (2, N), dostal som {tm.shape}.')
    if tm.shape[1] != 2 and tm.shape[0] == 2:
        tm = tm.T
    tm = tm[np.isfinite(tm).all(axis=1)]
    tm = tm[np.argsort(tm[:, 0])]
    return tm

def unique_monotonic_xy(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    order = np.argsort(x)
    x = x[order]
    y = y[order]
    ux, inverse = np.unique(x, return_inverse=True)
    uy = np.zeros_like(ux, dtype=float)
    counts = np.zeros_like(ux, dtype=float)
    for idx, value in zip(inverse, y):
        uy[idx] += value
        counts[idx] += 1
    uy /= np.maximum(counts, 1)
    return (ux, uy)

def get_tempomap(song=None):
    data = get_song_data(song)
    return normalize_tempomap(data['tempomap'])

def score_to_audio(score_time, song=None):
    tm = get_tempomap(song=song)
    x, y = unique_monotonic_xy(tm[:, 0], tm[:, 1])
    return np.interp(score_time, x, y, left=y[0], right=y[-1])

def audio_to_score(audio_time, song=None):
    tm = get_tempomap(song=song)
    x, y = unique_monotonic_xy(tm[:, 1], tm[:, 0])
    return np.interp(audio_time, x, y, left=y[0], right=y[-1])

def centers_to_edges(x):
    x = np.asarray(x, dtype=float)
    if len(x) == 0:
        return np.array([0.0, 1.0])
    if len(x) == 1:
        return np.array([x[0] - 0.5, x[0] + 0.5])
    mids = (x[:-1] + x[1:]) / 2
    first = x[0] - (mids[0] - x[0])
    last = x[-1] + (x[-1] - mids[-1])
    return np.concatenate([[first], mids, [last]])

def print_tempomap_range(song=None):
    data = get_song_data(song)
    tm = get_tempomap(song=data['song'])
    print(f"Skladba: {data['song']}")
    print(f'Rozsah čas notového zápisuu: {tm[:, 0].min():.3f} až {tm[:, 0].max():.3f}')
    print(f'Rozsah čas nahrávkyu: {tm[:, 1].min():.3f} až {tm[:, 1].max():.3f}')

print_tempomap_range(aktuálna_SONG)


## 5. Vizualizačné funkcie

Grafy sú pripravené tak, aby sa dali priamo použiť pri kontrole výsledkov a v prípade potreby aj exportovať ako obrázky do textu práce. Pri každom lokálnom úseku sa dá zobraziť tempomapa, porovnanie chroma príznakov, samostatné chroma matice a lokálna matica nákladov so zarovnávacou cestou.


In [ ]:
# @title Vizualizačné funkcie a analýza lokálneho úseku
LOCAL_PLOTS_DIR_NAME = 'local_plots'
LOCAL_PLOT_DPI = 150
SAVE_LOCAL_PLOTS_BY_DEFAULT = True
SHOW_LOCAL_PLOTS_BY_DEFAULT = True

CHROMA_LABELS = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
PITCH_LABELS = CHROMA_LABELS

AX_SCORE_TIME = 'čas v notovom zápise [doby]'
AX_AUDIO_TIME = 'čas v nahrávke [s]'
AX_SCORE_FRAME = 'rámec notového zápisu'
AX_AUDIO_FRAME = 'rámec nahrávky'
AX_PITCH_CLASS = 'tónová trieda'
CB_CHROMA = 'intenzita'
CB_COST = 'náklad'

TITLE_SCORE_CHROMA = 'Chroma reprezentácia notového zápisu'
TITLE_AUDIO_CHROMA = 'Chroma reprezentácia nahrávky'
TITLE_ALIGNED_CHROMAS = 'Porovnanie chroma reprezentácií po zarovnaní'
TITLE_LOCAL_COST_MATRIX = 'Matica nákladov so zarovnávacou cestou DTW'
TITLE_TEMPOMAP_SMOOTH = 'Vyhladená tempomapa'


def safe_filename_part(value):
    text = str(value)
    safe = ''.join(ch if (ch.isalnum() or ch in ('-', '_')) else '_' for ch in text)
    while '__' in safe:
        safe = safe.replace('__', '_')
    return safe.strip('_') or 'plot'


def format_time_for_filename(value):
    value = float(value)
    text = f'{value:.3f}'
    return text.replace('-', 'm').replace('.', 'p')


def get_local_plot_dir(data, save_dir=None):
    if save_dir is None:
        plot_dir = Path(data['song_dir']) / LOCAL_PLOTS_DIR_NAME
    else:
        plot_dir = Path(save_dir)
    plot_dir.mkdir(parents=True, exist_ok=True)
    return plot_dir


def make_local_plot_path(data, plot_name, audio_start, audio_end, save_dir=None):
    plot_dir = get_local_plot_dir(data, save_dir=save_dir)
    song = safe_filename_part(data['song'])
    plot_name = safe_filename_part(plot_name)
    start = format_time_for_filename(audio_start)
    end = format_time_for_filename(audio_end)
    return plot_dir / f'{song}__a{start}_to_{end}__{plot_name}.png'


def finish_local_plot(fig, save_path=None, show=True):
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=LOCAL_PLOT_DPI, bbox_inches='tight')
        print(f'Uložený graf: {save_path}')
    if show:
        plt.show()
    plt.close(fig)


def _format_chroma_axis(ax, ylabel=AX_PITCH_CLASS):
    ax.set_yticks(range(12))
    ax.set_yticklabels(CHROMA_LABELS)
    ax.set_ylabel(ylabel)


def _plot_path_overlay(ax, x, y):
    ax.plot(x, y, color='white', linewidth=3.2, alpha=0.95, zorder=3)
    ax.plot(x, y, color='red', linewidth=1.8, alpha=1.0, zorder=4)


def _extent_from_times(times, y_min=-0.5, y_max=11.5):
    times = np.asarray(times, dtype=float).squeeze()
    if len(times) == 0:
        return None
    if len(times) == 1:
        return [float(times[0]) - 0.5, float(times[0]) + 0.5, y_min, y_max]
    return [float(times[0]), float(times[-1]), y_min, y_max]


def normalize_columns(X, eps=1e-12):
    X = np.asarray(X, dtype=float)
    norms = np.linalg.norm(X, axis=0, keepdims=True)
    return X / np.maximum(norms, eps)


def slice_indices_by_time(times, start, end):
    times = np.asarray(times, dtype=float).squeeze()
    mask = (times >= start) & (times <= end)
    idx = np.where(mask)[0]
    if len(idx) == 0:
        return (None, None, mask)
    return (int(idx[0]), int(idx[-1] + 1), mask)


def normalize_path_frames(path_frames):
    if path_frames is None:
        return None
    pf = np.asarray(path_frames)
    if pf.ndim != 2:
        return None
    if pf.shape[1] >= 2:
        pf = pf[:, :2]
    elif pf.shape[0] >= 2:
        pf = pf[:2, :].T
    else:
        return None
    pf = pf[np.isfinite(pf).all(axis=1)]
    if len(pf) == 0:
        return None
    return np.rint(pf).astype(int)


def plot_chroma_on_axis(chroma, x_centers, title, xlabel, save_path=None, show=True):
    if chroma is None:
        print(f"Preskakujem '{title}', lebo chroma dáta nie sú načítané.")
        return

    chroma = orient_chroma(chroma)
    x_centers = np.asarray(x_centers, dtype=float).squeeze()

    if chroma.shape[1] == 0 or len(x_centers) == 0:
        print(f"Preskakujem '{title}', lebo vybraný úsek je prázdny.")
        return
    if len(x_centers) != chroma.shape[1]:
        print(f"Preskakujem '{title}', lebo počet časov ({len(x_centers)}) nesedí s počtom rámcov ({chroma.shape[1]}).")
        return

    fig, ax = plt.subplots(figsize=(10, 4))
    image = ax.imshow(
        chroma,
        origin='lower',
        aspect='auto',
        interpolation='nearest',
        extent=_extent_from_times(x_centers),
    )
    ax.set_xlabel(xlabel)
    _format_chroma_axis(ax)
    ax.set_title(title)
    fig.colorbar(image, ax=ax, label=CB_CHROMA)
    fig.tight_layout()
    finish_local_plot(fig, save_path=save_path, show=show)


def get_score_chroma_on_audio_time(data):
    audio_times = data.get('audio_times')
    audio_chroma = data.get('audio_chroma')
    if audio_times is None:
        print('Chroma notového zápisu premietnutá do času nahrávky sa nedá pripraviť, lebo chýba audio_times.npy.')
        return None
    audio_times = np.asarray(audio_times, dtype=float).squeeze()
    expected_frames = len(audio_times)

    score_chroma_on_audio = data.get('score_chroma_on_audio')
    if score_chroma_on_audio is not None:
        score_chroma_on_audio = orient_chroma(score_chroma_on_audio)
        if score_chroma_on_audio.shape[1] == expected_frames:
            return score_chroma_on_audio
        if audio_chroma is not None and score_chroma_on_audio.shape[1] == audio_chroma.shape[1]:
            return score_chroma_on_audio
        print('Pozn.: uložený score_chroma_on_audio_time.npy nemá rovnaký počet rámcov ako audio_times.npy, preto ho prepočítam z tempomapy.')

    score_chroma = data.get('score_chroma')
    score_times = data.get('score_times')
    tempomap = data.get('tempomap')
    if score_chroma is None or score_times is None or tempomap is None:
        print('Chroma notového zápisu premietnutá do času nahrávky sa nedá pripraviť, lebo chýba score_chroma.npy, score_times.npy alebo tempomapa.')
        return None

    score_chroma_on_audio = compute_score_chroma_on_audio_time_from_tempomap(
        score_times=score_times,
        score_chroma=score_chroma,
        audio_times=audio_times,
        tempomap=tempomap,
        normalize=True,
    )
    data['score_chroma_on_audio'] = score_chroma_on_audio
    return score_chroma_on_audio


def plot_aligned_chromas_comparison(data, audio_start, audio_end, save_path=None, show=True):
    audio_chroma = data.get('audio_chroma')
    audio_times = data.get('audio_times')
    if audio_chroma is None or audio_times is None:
        print('Spoločný chroma graf sa nedá zobraziť, lebo chýba audio_chroma.npy alebo audio_times.npy.')
        return

    score_chroma_on_audio = get_score_chroma_on_audio_time(data)
    if score_chroma_on_audio is None:
        return

    audio_chroma = orient_chroma(audio_chroma)
    score_chroma_on_audio = orient_chroma(score_chroma_on_audio)
    audio_times = np.asarray(audio_times, dtype=float).squeeze()

    n_frames = min(audio_chroma.shape[1], score_chroma_on_audio.shape[1], len(audio_times))
    audio_chroma = audio_chroma[:, :n_frames]
    score_chroma_on_audio = score_chroma_on_audio[:, :n_frames]
    audio_times = audio_times[:n_frames]

    mask = (audio_times >= audio_start) & (audio_times <= audio_end)
    if not np.any(mask):
        print('Spoločný chroma graf sa nedá zobraziť, lebo vybraný audio úsek je mimo časovej osi.')
        return

    t = audio_times[mask]
    score_segment = score_chroma_on_audio[:, mask]
    audio_segment = audio_chroma[:, mask]
    extent = _extent_from_times(t)

    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True, sharey=True)
    score_image = axes[0].imshow(
        score_segment,
        origin='lower',
        aspect='auto',
        interpolation='nearest',
        extent=extent,
    )
    audio_image = axes[1].imshow(
        audio_segment,
        origin='lower',
        aspect='auto',
        interpolation='nearest',
        extent=extent,
    )

    axes[0].set_title('Chroma notového zápisu premietnutá do čas nahrávkyu')
    axes[1].set_title('Chroma nahrávky')
    axes[1].set_xlabel(AX_AUDIO_TIME)

    for ax in axes:
        _format_chroma_axis(ax)

    fig.colorbar(score_image, ax=axes[0], label=CB_CHROMA)
    fig.colorbar(audio_image, ax=axes[1], label=CB_CHROMA)
    fig.tight_layout()
    finish_local_plot(fig, save_path=save_path, show=show)


def compute_score_chroma_on_audio_time_from_tempomap(score_times, score_chroma, audio_times, tempomap, normalize=True):
    score_times = np.asarray(score_times, dtype=float).squeeze()
    audio_times = np.asarray(audio_times, dtype=float).squeeze()
    score_chroma = orient_chroma(score_chroma)
    tempomap = normalize_tempomap(tempomap)

    score_times_on_audio = np.interp(
        score_times,
        *unique_monotonic_xy(tempomap[:, 0], tempomap[:, 1]),
        left=tempomap[:, 1].min(),
        right=tempomap[:, 1].max(),
    )
    order = np.argsort(score_times_on_audio)
    x = score_times_on_audio[order]
    Y = score_chroma[:, order].T
    x_unique, unique_idx = np.unique(x, return_index=True)
    Y_unique = Y[unique_idx]

    warped = np.zeros((12, len(audio_times)), dtype=float)
    for pc in range(12):
        warped[pc, :] = np.interp(audio_times, x_unique, Y_unique[:, pc], left=0.0, right=0.0)
    if normalize:
        warped = normalize_columns(warped)
    return warped


def plot_tempomap_segment(data, audio_start, audio_end, score_start, score_end, save_path=None, show=True):
    n_points = 600
    score_grid = np.linspace(score_start, score_end, n_points)
    audio_grid = score_to_audio(score_grid, song=data['song'])

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(score_grid, audio_grid, linewidth=1.8)
    ax.set_xlabel(AX_SCORE_TIME)
    ax.set_ylabel(AX_AUDIO_TIME)
    ax.set_title(f'{TITLE_TEMPOMAP_SMOOTH} – vybraný úsek')
    ax.set_xlim(score_start, score_end)
    ax.set_ylim(audio_start, audio_end)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    finish_local_plot(fig, save_path=save_path, show=show)


def plot_cost_matrix_with_path(data, audio_start, audio_end, score_start, score_end, save_path=None, show=True):
    audio_chroma = data['audio_chroma']
    score_chroma = data['score_chroma']
    audio_times = data['audio_times']
    score_times = data['score_times']

    if audio_chroma is None or score_chroma is None:
        print('Nákladová matica sa nedá zobraziť, lebo chýba audio_chroma.npy alebo score_chroma.npy.')
        return
    if audio_times is None or score_times is None:
        print('Nákladová matica sa nedá zobraziť, lebo chýba audio_times.npy alebo score_times.npy.')
        return

    audio_chroma = orient_chroma(audio_chroma)
    score_chroma = orient_chroma(score_chroma)
    audio_times = np.asarray(audio_times, dtype=float).squeeze()
    score_times = np.asarray(score_times, dtype=float).squeeze()

    a0, a1, _ = slice_indices_by_time(audio_times, audio_start, audio_end)
    s0, s1, _ = slice_indices_by_time(score_times, score_start, score_end)
    if a0 is None:
        print('Nákladová matica sa nedá zobraziť, lebo vybraný audio úsek je mimo čas nahrávkyovej osi.')
        return
    if s0 is None:
        print('Nákladová matica sa nedá zobraziť, lebo vypočítaný score úsek je mimo čas notového zápisuovej osi.')
        return

    audio_seg = normalize_columns(audio_chroma[:, a0:a1])
    score_seg = normalize_columns(score_chroma[:, s0:s1])
    local_cost = 1.0 - np.clip(score_seg.T @ audio_seg, -1.0, 1.0)

    local_path = None
    pf = normalize_path_frames(data.get('path_frames'))
    if pf is not None:
        mask = (pf[:, 0] >= s0) & (pf[:, 0] < s1) & (pf[:, 1] >= a0) & (pf[:, 1] < a1)
        local_path = pf[mask]

    fig, ax = plt.subplots(figsize=(8, 6))
    extent = [
        float(score_times[s0]),
        float(score_times[s1 - 1]),
        float(audio_times[a0]),
        float(audio_times[a1 - 1]),
    ]
    image = ax.imshow(
        local_cost.T,
        origin='lower',
        aspect='auto',
        interpolation='nearest',
        extent=extent,
    )

    if local_path is not None and len(local_path) > 0:
        path_score_times = score_times[local_path[:, 0]]
        path_audio_times = audio_times[local_path[:, 1]]
        _plot_path_overlay(ax, path_score_times, path_audio_times)
    else:
        print('Pozn.: Pre tento úsek nemám dostupnú DTW cestu. Skontroluj, či existuje path_frames.npy.')

    ax.set_xlabel(AX_SCORE_TIME)
    ax.set_ylabel(AX_AUDIO_TIME)
    ax.set_title(f'{TITLE_LOCAL_COST_MATRIX}')
    fig.colorbar(image, ax=ax, label=CB_COST)
    fig.tight_layout()
    finish_local_plot(fig, save_path=save_path, show=show)

    print(f'Lokálny tvar nákladovej matice: {local_cost.shape}')
    print(f'Počet bodov DTW cesty v úseku: {(0 if local_path is None else len(local_path))}')


def show_alignment_segment(
    song=None,
    audio_start=10.0,
    audio_end=130.0,
    show_audio_player=False,
    show_chromas=True,
    show_aligned_chromas=True,
    show_tempomap=True,
    show_tempomap_detail=None,
    show_cost_matrix=True,
    save_plots=SAVE_LOCAL_PLOTS_BY_DEFAULT,
    save_dir=None,
    show_plots=SHOW_LOCAL_PLOTS_BY_DEFAULT,
):
    if show_tempomap_detail is not None:
        show_tempomap = show_tempomap_detail

    data = get_song_data(song)
    if audio_end <= audio_start:
        raise ValueError('audio_end musí byť väčšie ako audio_start.')

    score_start = float(audio_to_score(audio_start, song=data['song']))
    score_end = float(audio_to_score(audio_end, song=data['song']))
    if score_end < score_start:
        score_start, score_end = (score_end, score_start)

    print('Vybraný úsek')
    print(f" - skladba: {data['song']}")
    print(f' - čas nahrávky: {audio_start:.3f} s až {audio_end:.3f} s')
    print(f' - zodpovedajúci čas notového zápisu: {score_start:.3f} až {score_end:.3f}')
    if save_plots:
        print(f" - grafy sa ukladajú do: {get_local_plot_dir(data, save_dir=save_dir)}")

    if show_audio_player:
        audio_path = data['audio_path']
        if audio_path and Path(audio_path).exists():
            display(Audio(filename=str(audio_path)))
        else:
            print('Audio prehrávač sa nezobrazí, pretože WAV súbor nie je dostupný v analysis priečinku.')

    if show_tempomap:
        save_path = make_local_plot_path(data, 'tempomap_segment', audio_start, audio_end, save_dir=save_dir) if save_plots else None
        plot_tempomap_segment(data, audio_start, audio_end, score_start, score_end, save_path=save_path, show=show_plots)

    if show_aligned_chromas:
        save_path = make_local_plot_path(data, 'aligned_chromas_comparison', audio_start, audio_end, save_dir=save_dir) if save_plots else None
        plot_aligned_chromas_comparison(data, audio_start, audio_end, save_path=save_path, show=show_plots)

    if show_chromas:
        audio_chroma = data['audio_chroma']
        audio_times = data['audio_times']
        score_chroma = data['score_chroma']
        score_times = data['score_times']

        if audio_chroma is not None and audio_times is not None:
            audio_chroma = orient_chroma(audio_chroma)
            audio_times_arr = np.asarray(audio_times, dtype=float).squeeze()
            audio_mask = (audio_times_arr >= audio_start) & (audio_times_arr <= audio_end)
            save_path = make_local_plot_path(data, 'audio_chroma_segment', audio_start, audio_end, save_dir=save_dir) if save_plots else None
            plot_chroma_on_axis(
                audio_chroma[:, audio_mask],
                audio_times_arr[audio_mask],
                f'{TITLE_AUDIO_CHROMA} ({data["song"]})',
                AX_AUDIO_TIME,
                save_path=save_path,
                show=show_plots,
            )

        if score_chroma is not None and score_times is not None:
            score_chroma = orient_chroma(score_chroma)
            if data.get('score_chroma_on_audio') is not None and audio_times is not None:
                audio_times_arr = np.asarray(audio_times, dtype=float).squeeze()
                audio_mask = (audio_times_arr >= audio_start) & (audio_times_arr <= audio_end)
                score_chroma_on_audio = orient_chroma(data['score_chroma_on_audio'])
                save_path = make_local_plot_path(data, 'score_chroma_on_audio_time_segment', audio_start, audio_end, save_dir=save_dir) if save_plots else None
                plot_chroma_on_axis(
                    score_chroma_on_audio[:, audio_mask],
                    audio_times_arr[audio_mask],
                    f'{TITLE_SCORE_CHROMA} premietnutá do času nahrávky ({data["song"]})',
                    AX_AUDIO_TIME,
                    save_path=save_path,
                    show=show_plots,
                )
            else:
                score_times_arr = np.asarray(score_times, dtype=float).squeeze()
                score_mask = (score_times_arr >= score_start) & (score_times_arr <= score_end)
                selected_score_times = score_times_arr[score_mask]
                mapped_score_times = score_to_audio(selected_score_times, song=data['song'])
                save_path = make_local_plot_path(data, 'score_chroma_mapped_to_audio_time_segment', audio_start, audio_end, save_dir=save_dir) if save_plots else None
                plot_chroma_on_axis(
                    score_chroma[:, score_mask],
                    mapped_score_times,
                    f'{TITLE_SCORE_CHROMA} premietnutá do času nahrávky ({data["song"]})',
                    AX_AUDIO_TIME,
                    save_path=save_path,
                    show=show_plots,
                )

    if show_cost_matrix:
        save_path = make_local_plot_path(data, 'local_cost_matrix_with_path', audio_start, audio_end, save_dir=save_dir) if save_plots else None
        plot_cost_matrix_with_path(
            data=data,
            audio_start=audio_start,
            audio_end=audio_end,
            score_start=score_start,
            score_end=score_end,
            save_path=save_path,
            show=show_plots,
        )


## 6. Analýza lokálneho úseku

Hlavná funkcia `show_alignment_segment` zobrazí konkrétny interval nahrávky a pomocou tempomapy dopočíta zodpovedajúci úsek v notovom zápise. Táto funkcia generuje všetky lokálne grafy používané pri interpretácii výsledkov.

## 7. Globálny prehľad tempomapy

Táto bunka vykreslí celú tempomapu pre zvolené skladby. Globálny graf je vhodný na rýchlu kontrolu, či mapovanie rastie plynulo a či neobsahuje výrazné skoky mimo očakávaných miest.


In [ ]:
# @title Globálny graf tempomapy

def plot_full_tempomap(song=None, save_path=None, show=True):
    data = get_song_data(song)
    tm = get_tempomap(data['song'])

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(tm[:, 0], tm[:, 1], linewidth=1.5)
    ax.set_xlabel(AX_SCORE_TIME)
    ax.set_ylabel(AX_AUDIO_TIME)
    ax.set_title(f'{TITLE_TEMPOMAP_SMOOTH} – {data["song"]}')
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    finish_local_plot(fig, save_path=save_path, show=show)


GLOBAL_TEMPOMAP_SONGS = [song for song in ['aka-si-mi-krasna', 'palenocka', 'palenocka-live', 'chopin'] if song in SONGS]
for song in GLOBAL_TEMPOMAP_SONGS:
    data = get_song_data(song)
    save_path = get_local_plot_dir(data) / f'{safe_filename_part(song)}__full_tempomap.png'
    plot_full_tempomap(song=song, save_path=save_path, show=True)


## 8. Ukážky lokálneho zarovnania – stabilné a problematické úseky

Nasledujúce úseky sú pripravené ako podklady pre experimentálnu kapitolu. Pri každom úseku sa uložia rovnaké typy grafov, aby bolo možné porovnať rozdiel medzi spoľahlivým zarovnaním a miestami, kde sa prejavujú lokálne chyby spôsobené opakovanými alebo menej výraznými tónmi.


In [ ]:
# @title Definícia analyzovaných úsekov
ANALYZOVANE_USEKY = [
    {
        'song': 'aka-si-mi-krasna',
        'audio_start': 0,
        'audio_end': 14,
        'note': 'Začiatok skladby – kontrola globálne správneho zarovnania.',
    },
    {
        'song': 'aka-si-mi-krasna',
        'audio_start': 60,
        'audio_end': 74,
        'note': 'Neskorší úsek rovnakej skladby – kontrola stability mapovania v čase.',
    },
    {
        'song': 'palenocka-live',
        'audio_start': 88,
        'audio_end': 95,
        'note': 'Živá nahrávka skladby Pálenôčka – lokálne problematický úsek.',
    },
    {
        'song': 'palenocka',
        'audio_start': 101,
        'audio_end': 108,
        'note': 'Porovnateľný úsek v syntetickej alebo stabilnejšej verzii skladby Pálenôčka.',
    },
]

for i, segment in enumerate(ANALYZOVANE_USEKY, start=1):
    print(f"{i}. {segment['song']} ({segment['audio_start']}–{segment['audio_end']} s): {segment['note']}")
    if segment['song'] not in SONGS:
        print('   Pozn.: skladba momentálne nie je dostupná v načítaných výstupoch.')


In [ ]:
# @title Vykreslenie analyzovaných úsekov
for segment in ANALYZOVANE_USEKY:
    if segment['song'] not in SONGS:
        continue

    print('\n' + '=' * 90)
    print(segment['note'])
    show_alignment_segment(
        song=segment['song'],
        audio_start=segment['audio_start'],
        audio_end=segment['audio_end'],
        show_audio_player=False,
        show_chromas=True,
        show_aligned_chromas=True,
        show_tempomap=True,
        show_cost_matrix=True,
        save_plots=True,
        show_plots=True,
    )


## 9. Ako čítať vygenerované grafy

- **Tempomapa** ukazuje, ako sa čas v notovom zápise premieta do času nahrávky. Približne lineárny priebeh znamená stabilné tempo; zmeny sklonu zodpovedajú lokálnemu zrýchleniu alebo spomaleniu.
- **Porovnanie chroma reprezentácií po zarovnaní** umožňuje vizuálne skontrolovať, či sa tónové triedy notového zápisu po premietnutí do času nahrávky podobajú extrahovaným príznakom zo zvuku.
- **Matica nákladov so zarovnávacou cestou DTW** zobrazuje lokálnu podobnosť rámcov. Zarovnávacia cesta by mala prechádzať oblasťami nízkych nákladov a mala by mať plynulý priebeh.
- **Samostatné chroma grafy** sú vhodné na kontrolu, či chyba pochádza z príznakov, z mapovania tempomapy alebo z menej jednoznačného hudobného obsahu v danom úseku.


## 10. Poznámky k použitiu v texte práce

Obrázky sa ukladajú do priečinka `local_plots` pri konkrétnej skladbe. Pri vkladaní do bakalárskej práce odporúčam používať iba najvýstižnejšie grafy, nie všetky výstupy notebooku. Pre každý vybraný príklad stačí spravidla ukázať tempomapu alebo lokálnu maticu nákladov a v texte stručne vysvetliť, čo graf potvrdzuje.

Notebook je určený ako reprodukovateľný doplnok k implementácii: po opätovnom spustení hlavnej pipeline a následnom spustení tohto notebooku by mali vzniknúť rovnaké typy grafov v rovnakom pomenovaní.
